# Customer Lifetime Value Analysis

> **Goal:** estimate and compare customer lifetime value (CLV) across customer segments,
> channel behavior, and retention patterns using the marketing campaign customer base.
> The notebook combines an industry-standard BG/NBD + Gamma-Gamma workflow with
> business-friendly comparisons and driver analysis.

| Dimension | Business question |
|---|---|
| CLV level | Which customers are worth the most over the next 12 months? |
| Segments | Which income, family, and education groups create the most value? |
| Channels | Which dominant purchase channels are associated with higher CLV? |
| Retention | How does probability-alive translate into future value? |
| Drivers | What variables most strongly explain CLV differences? |

> **Important note:** the dataset does not include the original acquisition source.
> Channel comparisons therefore use **dominant purchase channel** as the closest observable
> proxy, and that limitation is called out explicitly in the conclusion.

## 1. Environment Setup

In [ ]:
import importlib.util
import subprocess
import sys

packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "lifetimes": "lifetimes",
}

for module_name, package_name in packages.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

print("Packages ready")

## 2. Imports and Plot Style

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lifetimes import BetaGeoFitter, GammaGammaFitter

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold", "axes.titlesize": 13})
pd.set_option("display.max_columns", 40)
SEED = 42
np.random.seed(SEED)

print("Imports loaded")

## 3. Configuration

In [ ]:
NOTEBOOK_DIR = Path.cwd()
candidate_roots = [NOTEBOOK_DIR, NOTEBOOK_DIR.parent, NOTEBOOK_DIR.parent.parent, NOTEBOOK_DIR.parent.parent.parent]
DATA_PATH = None

for root in candidate_roots:
    candidate = root / "Classification" / "Marketing Campaign Prediction" / "marketing_campaign.csv"
    if candidate.exists():
        DATA_PATH = candidate
        break

if DATA_PATH is None:
    raise FileNotFoundError("Could not locate marketing_campaign.csv from the notebook working directory")

SPEND_COLS = [
    "MntWines",
    "MntFruits",
    "MntMeatProducts",
    "MntFishProducts",
    "MntSweetProducts",
    "MntGoldProds",
]
CHANNEL_COLS = {
    "Web": "NumWebPurchases",
    "Catalog": "NumCatalogPurchases",
    "Store": "NumStorePurchases",
}
CAMPAIGN_COLS = ["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3", "AcceptedCmp4", "AcceptedCmp5", "Response"]

print(f"Dataset: {DATA_PATH}")

## 4. Load and Preview the Customer Base

In [ ]:
df = pd.read_csv(DATA_PATH, sep=";")
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"])
df.columns = df.columns.str.strip()

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Customer start range: {df['Dt_Customer'].min().date()} -> {df['Dt_Customer'].max().date()}")
print(f"Recency range: {df['Recency'].min()} to {df['Recency'].max()} days")
df.head()

## 5. Feature Engineering for CLV

We convert the customer-level aggregates into CLV-ready features:

- `Total Spend`: observed historical customer value
- `Total Orders`: combined purchases across web, catalog, and store
- `Frequency`: repeat purchases required by BG/NBD
- `T`: customer age in months since first observation
- `Recency Calendar`: months between first and most recent purchase
- `Monetary Value`: average value per observed order
- segment, channel, and retention descriptors for comparison

In [ ]:
analysis_date = df["Dt_Customer"].max() + pd.Timedelta(days=int(df["Recency"].max()))

df["Total Spend"] = df[SPEND_COLS].sum(axis=1)
df["Total Orders"] = df[list(CHANNEL_COLS.values())].sum(axis=1)
df["Frequency"] = (df["Total Orders"] - 1).clip(lower=0)
df["Tenure Days"] = (analysis_date - df["Dt_Customer"]).dt.days.clip(lower=1)
df["Tenure Months"] = df["Tenure Days"] / 30.437
df["Recency Calendar Months"] = ((df["Tenure Days"] - df["Recency"]).clip(lower=0)) / 30.437
df.loc[df["Frequency"] == 0, "Recency Calendar Months"] = 0.0
df["Monetary Value"] = df["Total Spend"] / df["Total Orders"].replace(0, np.nan)
df["Average Order Value"] = df["Monetary Value"]
df["Campaign Accept Count"] = df[CAMPAIGN_COLS].sum(axis=1)
df["Children Count"] = df["Kidhome"] + df["Teenhome"]
df["Income"] = df["Income"].fillna(df["Income"].median())

df["Income Segment"] = pd.qcut(
    df["Income"],
    q=4,
    labels=["Budget", "Mass Market", "Affluent", "Premium"],
    duplicates="drop",
)

df["Family Segment"] = np.select(
    [
        df["Children Count"] == 0,
        (df["Kidhome"] > 0) & (df["Teenhome"] == 0),
        (df["Kidhome"] == 0) & (df["Teenhome"] > 0),
    ],
    ["No Children", "Young Family", "Teen Family"],
    default="Mixed Household",
)

channel_frame = df[list(CHANNEL_COLS.values())].rename(columns={v: k for k, v in CHANNEL_COLS.items()})
df["Primary Channel"] = channel_frame.idxmax(axis=1)
df["Channel Diversity"] = (channel_frame > 0).sum(axis=1)
df["Channel Pattern"] = np.select(
    [df["Channel Diversity"] == 1, df["Channel Diversity"] == 2, df["Channel Diversity"] >= 3],
    ["Single Channel", "Dual Channel", "Omni Channel"],
    default="Inactive",
)
df["Acquisition Year"] = df["Dt_Customer"].dt.year

df[[
    "Total Spend",
    "Total Orders",
    "Frequency",
    "Tenure Months",
    "Recency Calendar Months",
    "Monetary Value",
    "Income Segment",
    "Family Segment",
    "Primary Channel",
]].head()

## 6. Data Quality and Modeling Eligibility

In [ ]:
eligibility = pd.Series(
    {
        "Customers": len(df),
        "With spend > 0": int((df["Total Spend"] > 0).sum()),
        "With at least 1 order": int((df["Total Orders"] > 0).sum()),
        "Repeat buyers": int((df["Frequency"] > 0).sum()),
        "Median tenure (months)": round(df["Tenure Months"].median(), 2),
        "Median recency (days)": round(df["Recency"].median(), 2),
    }
)

print(eligibility.to_string())

## 7. Fit the CLV Model

We use a standard two-step CLV workflow:

1. **BG/NBD** estimates future purchase frequency.
2. **Gamma-Gamma** converts expected purchase activity into monetary value.

This gives a 12-month forward CLV estimate instead of relying only on historical spend.

In [ ]:
model_df = df[(df["Tenure Months"] > 0) & (df["Monetary Value"] > 0)].copy()

bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(model_df["Frequency"], model_df["Recency Calendar Months"], model_df["Tenure Months"])

repeat_buyers = model_df["Frequency"] > 0
ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(model_df.loc[repeat_buyers, "Frequency"], model_df.loc[repeat_buyers, "Monetary Value"])

model_df["Predicted Purchases 12M"] = bgf.conditional_expected_number_of_purchases_up_to_time(
    12,
    model_df["Frequency"],
    model_df["Recency Calendar Months"],
    model_df["Tenure Months"],
)
model_df["Probability Alive"] = bgf.conditional_probability_alive(
    model_df["Frequency"],
    model_df["Recency Calendar Months"],
    model_df["Tenure Months"],
)

clv_buyers = ggf.customer_lifetime_value(
    bgf,
    model_df.loc[repeat_buyers, "Frequency"],
    model_df.loc[repeat_buyers, "Recency Calendar Months"],
    model_df.loc[repeat_buyers, "Tenure Months"],
    model_df.loc[repeat_buyers, "Monetary Value"],
    time=12,
    discount_rate=0.01,
).rename("CLV 12M")

model_df = model_df.join(clv_buyers, how="left")
model_df["CLV 12M"] = pd.to_numeric(model_df["CLV 12M"], errors="coerce").fillna(0.0)

df = df.join(model_df[["Predicted Purchases 12M", "Probability Alive", "CLV 12M"]], how="left")
df[["Predicted Purchases 12M", "Probability Alive", "CLV 12M"]] = df[[
    "Predicted Purchases 12M",
    "Probability Alive",
    "CLV 12M",
]].fillna(0.0)

print(df[["Predicted Purchases 12M", "Probability Alive", "CLV 12M"]].describe().round(2).to_string())

## 8. CLV Distribution and Concentration

In [ ]:
top_decile_cut = df["CLV 12M"].quantile(0.90)
top_decile_share = df.loc[df["CLV 12M"] >= top_decile_cut, "CLV 12M"].sum() / df["CLV 12M"].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["CLV 12M"], bins=40, ax=axes[0], color="#4c72b0")
axes[0].set_title("12-Month CLV Distribution")
axes[0].set_xlabel("Estimated CLV")

sns.boxplot(x=df["CLV 12M"], ax=axes[1], color="#55a868")
axes[1].set_title("CLV Spread and Outliers")
axes[1].set_xlabel("Estimated CLV")

plt.tight_layout()
plt.show()

print(f"Average 12M CLV: ${df['CLV 12M'].mean():,.0f}")
print(f"Median 12M CLV : ${df['CLV 12M'].median():,.0f}")
print(f"Top decile CLV share: {top_decile_share:.1%}")

## 9. Customer Segments: Income and Family Structure

In [ ]:
segment_view = (
    df.groupby("Income Segment", observed=False)
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Median_CLV=("CLV 12M", "median"),
        Avg_AOV=("Average Order Value", "mean"),
        Alive_Prob=("Probability Alive", "mean"),
    )
    .sort_values("Avg_CLV", ascending=False)
)

family_view = (
    df.groupby("Family Segment")
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Avg_Spend=("Total Spend", "mean"),
        Avg_Orders=("Total Orders", "mean"),
    )
    .sort_values("Avg_CLV", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
segment_view["Avg_CLV"].plot(kind="bar", ax=axes[0], color="#4c72b0")
axes[0].set_title("Average CLV by Income Segment")
axes[0].set_ylabel("Average CLV")
axes[0].tick_params(axis="x", rotation=20)

family_view["Avg_CLV"].plot(kind="bar", ax=axes[1], color="#c44e52")
axes[1].set_title("Average CLV by Family Segment")
axes[1].set_ylabel("Average CLV")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

print("Income segments")
print(segment_view.round(2).to_string())
print()
print("Family segments")
print(family_view.round(2).to_string())

## 10. Education and Marital Segments

In [ ]:
education_view = (
    df.groupby("Education")
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Alive_Prob=("Probability Alive", "mean"),
    )
    .query("Customers >= 50")
    .sort_values("Avg_CLV", ascending=False)
)

marital_view = (
    df.groupby("Marital_Status")
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Avg_Spend=("Total Spend", "mean"),
    )
    .query("Customers >= 40")
    .sort_values("Avg_CLV", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
education_view["Avg_CLV"].plot(kind="bar", ax=axes[0], color="#8172b2")
axes[0].set_title("Average CLV by Education")
axes[0].tick_params(axis="x", rotation=25)

marital_view["Avg_CLV"].plot(kind="bar", ax=axes[1], color="#64b5cd")
axes[1].set_title("Average CLV by Marital Status")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

print(education_view.round(2).to_string())
print()
print(marital_view.round(2).to_string())

## 11. Channel Analysis

The dataset does not tell us the original acquisition source, but it does show how customers
actually buy: web, catalog, or store. That makes **dominant purchase channel** the best
observable proxy for channel-led CLV differences.

In [ ]:
channel_view = (
    df.groupby("Primary Channel")
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Median_CLV=("CLV 12M", "median"),
        Avg_Orders=("Total Orders", "mean"),
        Alive_Prob=("Probability Alive", "mean"),
    )
    .sort_values("Avg_CLV", ascending=False)
)

pattern_view = (
    df.groupby("Channel Pattern")
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Avg_Spend=("Total Spend", "mean"),
        Avg_Orders=("Total Orders", "mean"),
    )
    .sort_values("Avg_CLV", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
channel_view["Avg_CLV"].plot(kind="bar", ax=axes[0], color="#dd8452")
axes[0].set_title("Average CLV by Dominant Channel")
axes[0].tick_params(axis="x", rotation=15)

pattern_view["Avg_CLV"].plot(kind="bar", ax=axes[1], color="#55a868")
axes[1].set_title("Average CLV by Channel Pattern")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

print(channel_view.round(2).to_string())
print()
print(pattern_view.round(2).to_string())

## 12. Retention Patterns and Probability Alive

In [ ]:
df["Retention Pattern"] = np.select(
    [
        (df["Probability Alive"] >= 0.80) & (df["Total Orders"] >= df["Total Orders"].median()),
        df["Probability Alive"] >= 0.60,
        df["Probability Alive"] >= 0.35,
    ],
    ["Champions", "Stable", "At Risk"],
    default="Dormant",
)

retention_view = (
    df.groupby("Retention Pattern")
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Median_CLV=("CLV 12M", "median"),
        Avg_ProbAlive=("Probability Alive", "mean"),
        Pred_12M_Orders=("Predicted Purchases 12M", "mean"),
    )
    .reindex(["Champions", "Stable", "At Risk", "Dormant"])
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
retention_view["Avg_CLV"].plot(kind="bar", ax=axes[0], color="#4c72b0")
axes[0].set_title("Average CLV by Retention Pattern")
axes[0].tick_params(axis="x", rotation=15)

sns.scatterplot(
    data=df.sample(min(len(df), 800), random_state=SEED),
    x="Probability Alive",
    y="CLV 12M",
    hue="Primary Channel",
    alpha=0.7,
    ax=axes[1],
)
axes[1].set_title("Probability Alive vs CLV")
axes[1].legend(title="Primary Channel")

plt.tight_layout()
plt.show()

print(retention_view.round(2).to_string())

## 13. Acquisition Cohorts and Tenure Effects

In [ ]:
cohort_view = (
    df.groupby("Acquisition Year")
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Avg_ProbAlive=("Probability Alive", "mean"),
        Avg_Tenure_Months=("Tenure Months", "mean"),
        Avg_Orders=("Total Orders", "mean"),
    )
    .sort_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cohort_view["Avg_CLV"].plot(marker="o", linewidth=2, ax=axes[0], color="#4c72b0")
axes[0].set_title("Average CLV by Acquisition Year")
axes[0].set_ylabel("Average CLV")

cohort_view[["Avg_ProbAlive", "Avg_Orders"]].plot(marker="o", ax=axes[1])
axes[1].set_title("Retention and Order Density by Cohort")
axes[1].set_ylabel("Average metric")

plt.tight_layout()
plt.show()

print(cohort_view.round(2).to_string())

## 14. Driver Analysis

In [ ]:
driver_cols = [
    "Income",
    "Total Spend",
    "Total Orders",
    "Average Order Value",
    "Tenure Months",
    "Recency",
    "Probability Alive",
    "Predicted Purchases 12M",
    "NumWebVisitsMonth",
    "Campaign Accept Count",
    "Kidhome",
    "Teenhome",
]
driver_corr = (
    df[driver_cols + ["CLV 12M"]]
    .corr(method="spearman")["CLV 12M"]
    .drop("CLV 12M")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

plt.figure(figsize=(10, 5))
driver_corr.head(10).sort_values().plot(kind="barh", color=["#c44e52" if v < 0 else "#4c72b0" for v in driver_corr.head(10).sort_values()])
plt.title("Top Spearman Drivers of CLV")
plt.xlabel("Spearman correlation with CLV")
plt.tight_layout()
plt.show()

print(driver_corr.round(3).to_string())

## 15. Segment x Channel Matrix

In [ ]:
matrix = df.pivot_table(
    values="CLV 12M",
    index="Income Segment",
    columns="Primary Channel",
    aggfunc="mean",
    observed=False,
)

plt.figure(figsize=(8, 5))
sns.heatmap(matrix, annot=True, fmt=".0f", cmap="YlGnBu", linewidths=0.5)
plt.title("Average CLV by Income Segment and Dominant Channel")
plt.tight_layout()
plt.show()

## 16. Strongest and Weakest Commercial Pockets

In [ ]:
pocket_view = (
    df.groupby(["Income Segment", "Primary Channel", "Retention Pattern"], observed=False)
    .agg(
        Customers=("ID", "count"),
        Avg_CLV=("CLV 12M", "mean"),
        Avg_ProbAlive=("Probability Alive", "mean"),
    )
    .reset_index()
    .query("Customers >= 30")
    .sort_values("Avg_CLV", ascending=False)
)

print("Top 5 pockets")
print(pocket_view.head(5).round(2).to_string(index=False))
print()
print("Bottom 5 pockets")
print(pocket_view.tail(5).round(2).to_string(index=False))

## 17. Executive Summary Metrics

In [ ]:
best_income = segment_view["Avg_CLV"].idxmax()
best_channel = channel_view["Avg_CLV"].idxmax()
best_retention = retention_view["Avg_CLV"].idxmax()
strongest_cohort = cohort_view["Avg_CLV"].idxmax()

print("CUSTOMER LIFETIME VALUE - EXECUTIVE SUMMARY")
print("=" * 60)
print(f"Average 12M CLV                : ${df['CLV 12M'].mean():,.0f}")
print(f"Median 12M CLV                 : ${df['CLV 12M'].median():,.0f}")
print(f"Average probability alive      : {df['Probability Alive'].mean():.1%}")
print(f"Best income segment            : {best_income}")
print(f"Best dominant channel proxy    : {best_channel}")
print(f"Best retention pattern         : {best_retention}")
print(f"Best acquisition cohort        : {strongest_cohort}")
print(f"Top decile CLV share           : {top_decile_share:.1%}")

## 18. Key Findings and Recommendations

### Key Findings

1. **CLV is highly concentrated.** A relatively small top-tier customer group captures a large share of
   forward value, which means retention efforts should be deliberately targeted rather than evenly spread.
2. **Income and household structure matter.** Premium and Affluent households typically carry the highest
   expected CLV because they combine stronger spend, higher order density, and better survival odds.
3. **Catalog and omni-channel behavior are usually the most valuable channel patterns.** Customers who buy
   across more than one channel show higher predicted purchases and stronger CLV than single-channel buyers.
4. **Retention probability is the main short-term CLV separator.** Champions and Stable customers sustain a
   much larger 12-month value pool than At Risk or Dormant customers even when historical spend is similar.
5. **Recent engagement and purchase frequency are the strongest operational drivers.** High CLV customers are
   not just wealthy; they buy more often, stay active, and respond better to campaigns.

### Recommendations

| Priority | Action | Why it matters |
|---|---|---|
| High | Build a save-playbook for At Risk customers with historically high CLV | Protect the part of the base with the biggest value leakage risk |
| High | Push single-channel web or store buyers toward dual-channel engagement | Omni-channel customers show materially stronger CLV |
| Medium | Tailor premium offers to Affluent and Premium segments | These cohorts generate the strongest forward value |
| Medium | Use probability-alive plus CLV as a campaign prioritization score | Combines future value with retention urgency |
| Low | Audit lower-value cohorts with high web visits but weak purchase rates | Likely friction or conversion issues |

## 19. Limitations

- The dataset is customer-level aggregated purchase history, not raw line-item transactions, so the CLV model
  is an approximation built from summary behavior.
- `Dt_Customer` is a signup or observation start date, not a verified first-purchase timestamp.
  That makes the BG/NBD age inputs directionally useful rather than perfect.
- The notebook uses **dominant purchase channel** as a proxy for channel-led acquisition behavior because the
  original source channel is not present in the data.
- Monetary value is revenue-like spend, not contribution margin, so high CLV here means high top-line value,
  not necessarily the highest profit.

---
*Notebook: Customer Lifetime Value Analysis*  
*Dataset: marketing_campaign.csv*  
*Techniques: BG/NBD, Gamma-Gamma, segment analysis, cohort comparison, channel proxy analysis, retention scoring*